# Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENROUTER_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file"
print("Environment OK.")

Environment OK.


In [24]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver 
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langgraph.types import Command

In [3]:
model = init_chat_model(
    "openai/gpt-3.5-turbo",
    model_provider="openrouter",
)

In [4]:
response = model.invoke("Why do parrots talk? give me in 10 words")

In [5]:
# @title
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage,
    SystemMessage,
)


def pretty_print_chat(state):
    """
    Pretty print a LangChain agent state.

    Args:
        state: Dictionary returned by agent.invoke(...)
    """

    role_icons = {
        HumanMessage: "🧑 Human",
        AIMessage: "🤖 AI",
        ToolMessage: "🛠️ Tool",
        SystemMessage: "⚙️ System",
    }

    print("=" * 100)

    for i, message in enumerate(state["messages"], start=1):

        role = role_icons.get(type(message), type(message).__name__)

        print(f"\n{role} ({i})")
        print("-" * 100)

        if message.content:
            print(message.content)

        # Print tool calls made by the AI
        if isinstance(message, AIMessage) and message.tool_calls:
            print("\n🔧 Tool Calls:")
            for tool in message.tool_calls:
                print(f"   • Tool : {tool['name']}")
                print(f"     Args : {tool['args']}")
                print(f"     ID   : {tool['id']}")

        # Print tool outputs
        if isinstance(message, ToolMessage):
            print(f"\nTool Name    : {message.name}")
            print(f"Tool Call ID : {message.tool_call_id}")

    print("\n" + "=" * 100)

In [6]:
@tool
def check_showtimes(name:str) -> str:
    """Fucntion is built to give showtime for a movie
    Paramaters: name -> this is movie name"""
    showing = {
        "interstellar" : "everyday 7 PM",
        "basha": "Monday , Wednesday evening 5 PM",
        "nayagan": "everyday morning 10 AM"
    }
    return showing.get(name.lower(), "No shows available")

@tool
def cancel_booking(name:str) -> str:
    """"Built to cancel booking
    Paramaters: name -> this is movie name"""
    return f"Your seat for {name} is cancelled"

@tool 
def enquire_seat(name:str) -> str:
    """Built to provide list seat availability for movie enquired
    Paramaters: name -> this is movie name"""
    return f"5 seats available for movie {name}"

@tool
def cancel_policy() ->str:
    """This provides insighs about cancellation policy after booking movie tickets"""
    cancellation = """Customers can cancel movie tickets up to 2 hours before the showtime for a full refund.
Cancellations made within 2 hours of the showtime are not eligible for refunds.
Refunds, if applicable, will be processed to the original payment method within 5–7 business days."""
    return cancellation

In [7]:
saver = InMemorySaver()
thread_config = {"configurable": {"thread_id": "middle-demo-1"}}
agent = create_agent(model="openrouter:openai/gpt-3.5-turbo", 
                     tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
                     checkpointer=saver,
                     )

In [8]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "I want to book 3 tickets, can you check showtim for Basha movie also check is there any seat available to book?"}]
}, config=thread_config
)

In [9]:
response

{'messages': [HumanMessage(content='I want to book 3 tickets, can you check showtim for Basha movie also check is there any seat available to book?', additional_kwargs={}, response_metadata={}, id='19849e58-26a3-42b7-9f94-51184b389e71'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789006256-R5OxOQvxcZwkwqNrWM0n', 'created': 1789006256, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000162, 'cost_details': {'upstream_inference_completions_cost': 7.05e-05, 'upstream_inference_prompt_cost': 9.15e-05, 'upstream_inference_cost': 0.000162}}, id='lc_run--01a08915-14dd-70b2-a8a2-9e43a39c12c7-0', tool_calls=[{'name': 'check_showtimes', 'args': {'name': 'Basha'}, 'id': 'call_gL7jRbGgdbWMbVRjks5lxzlC', 'type': 'tool_call'}, {'name': 'enquire_seat', 'args': {'name': 'Basha'}, 'id': 'call_w1f8Z3xEoNmMiUCmw1B1Cf5f', 'type': 'tool_call'}], invalid_tool_calls=

In [10]:
agent = create_agent(model="openrouter:openai/gpt-3.5-turbo", 
                     tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
                     checkpointer=saver,
                     middleware=[
                         SummarizationMiddleware(
                             model= "openrouter:deepseek/deepseek-v4-flash-0731",
                             trigger=('tokens', 100),
                             keep=("messages", 2)
                         ),
                     ],
                     )

In [11]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "I want to book 3 tickets, can you check showtim for Basha movie also check is there any seat available to book?"}]
},config=thread_config
)

In [12]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nUser wants to book 3 tickets for the movie "Basha" and asked to check showtimes and seat availability before booking.\n\n## SUMMARY\n\n- The movie "Basha" is showing on Monday and Wednesday evening at 5 PM.\n- There are 5 seats available for booking.\n- The user was previously presented with these options and asked whether to proceed, but then repeated the request to check showtimes and seat availability for 3 tickets.\n- No showtime has been selected yet and the booking is not confirmed.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\n- Clearly present the available showtimes (Monday 5 PM, Wednesday 5 PM) and seat availability (5 seats) again.\n- Ask the user which showtime they prefer for the 3 tickets.\n- Once the user selects a showtime, proceed to book the 3 tickets.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='62f3d766-03b0-401d-9644-de3cfa3ff2d

In [13]:
response['messages'][-1].response_metadata

{'model_name': 'openai/gpt-3.5-turbo',
 'id': 'gen-1789006293-gI4yxqsdq8AOXzfU5k7j',
 'created': 1789006293,
 'object': 'chat.completion',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'openrouter',
 'cost': 0.000276,
 'cost_details': {'upstream_inference_completions_cost': 6.45e-05,
  'upstream_inference_prompt_cost': 0.0002115,
  'upstream_inference_cost': 0.000276}}

In [14]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "can you check was basha has showtime in Tuesday? let me know what is the cancellation policy?"}]
},config=thread_config
)

In [45]:
response

{'messages': [HumanMessage(content='can you check was basha has showtime in Tuesday? let me know what is the cancellation policy?', additional_kwargs={}, response_metadata={}, id='f2840fef-4d4e-4e62-9979-6d527fe3674e'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1788998817-gxoWsbNcsucU4DuYe2PO', 'created': 1788998817, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000153, 'cost_details': {'upstream_inference_completions_cost': 6.45e-05, 'upstream_inference_prompt_cost': 8.85e-05, 'upstream_inference_cost': 0.000153}}, id='lc_run--01a088a3-9505-7013-9281-f0efd616456c-0', tool_calls=[{'name': 'check_showtimes', 'args': {'name': 'basha'}, 'id': 'call_RUCb2RhckKtCa3nfBrAMSHPq', 'type': 'tool_call'}, {'name': 'cancel_policy', 'args': {}, 'id': 'call_vVrGyN7TGUCFsRtSxjZD3r66', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token

In [47]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "what are the movies has shows? on what days?"}]
},config=thread_config
)

In [48]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nUser originally wanted to book 3 tickets for the movie "Basha" and check showtimes/seat availability. The current query shifts to asking which movies have shows and on what days.\n\n## SUMMARY\n- Previously checked showtimes for "Basha": available Monday and Wednesday evening at 5 PM.\n- Seat availability for "Basha": 5 seats available.\n- No booking has been made yet; user has not selected a showtime.\n- The user has now asked a new question: "what are the movies has shows? on what days?" This indicates a broader inquiry about all available movies and their show schedules, moving beyond just "Basha."\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\n- Provide a list of all movies currently showing, along with their show days and times (if this data is available in the system).\n- Clarify whether the user still intends to book "Basha" tickets or is exploring other options.\n- If the user sel

In [50]:
pretty_print_chat(response)


🧑 Human (1)
----------------------------------------------------------------------------------------------------
Here is a summary of the conversation to date:

## SESSION INTENT
User originally wanted to book 3 tickets for the movie "Basha" and check showtimes/seat availability. The current query shifts to asking which movies have shows and on what days.

## SUMMARY
- Previously checked showtimes for "Basha": available Monday and Wednesday evening at 5 PM.
- Seat availability for "Basha": 5 seats available.
- No booking has been made yet; user has not selected a showtime.
- The user has now asked a new question: "what are the movies has shows? on what days?" This indicates a broader inquiry about all available movies and their show schedules, moving beyond just "Basha."

## ARTIFACTS
None.

## NEXT STEPS
- Provide a list of all movies currently showing, along with their show days and times (if this data is available in the system).
- Clarify whether the user still intends to book "Ba

# Human in the Loop(HITL)

In [17]:
agent = create_agent(model="openrouter:openai/gpt-3.5-turbo", 
                     tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
                     checkpointer=saver,
                     middleware=[
                         HumanInTheLoopMiddleware(
                             interrupt_on={
                                 "cancel_booking":True,
                             },
                             description_prefix="Tool execution pending approval",
                         ),
                     ],
                     )

In [18]:
hitl_config = {"configurable": {"thread_id": "middle-hitldemo-1"}}

In [19]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "can you cancel the booking A123"}]
},config=hitl_config
)

In [20]:
response

{'messages': [HumanMessage(content='can you cancel the booking A123', additional_kwargs={}, response_metadata={}, id='f9bc4892-d7fe-4c19-9711-b33f30ea5911'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789014554-jTJZU7cUqTnuHDdJy6Z3', 'created': 1789014554, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000104, 'cost_details': {'upstream_inference_completions_cost': 2.25e-05, 'upstream_inference_prompt_cost': 8.15e-05, 'upstream_inference_cost': 0.000104}}, id='lc_run--01a08993-b519-78e3-8945-e3e47f42c8e4-0', tool_calls=[{'name': 'cancel_booking', 'args': {'name': 'A123'}, 'id': 'call_SnGBOhQFDmyIKzKe9xPYPbfA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 15, 'total_tokens': 178, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})],
 '__

## Resume with approval decision

In [25]:
agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=hitl_config
)

{'messages': [HumanMessage(content='can you cancel the booking A123', additional_kwargs={}, response_metadata={}, id='f9bc4892-d7fe-4c19-9711-b33f30ea5911'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789014554-jTJZU7cUqTnuHDdJy6Z3', 'created': 1789014554, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000104, 'cost_details': {'upstream_inference_completions_cost': 2.25e-05, 'upstream_inference_prompt_cost': 8.15e-05, 'upstream_inference_cost': 0.000104}}, id='lc_run--01a08993-b519-78e3-8945-e3e47f42c8e4-0', tool_calls=[{'name': 'cancel_booking', 'args': {'name': 'A123'}, 'id': 'call_SnGBOhQFDmyIKzKe9xPYPbfA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 15, 'total_tokens': 178, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}),
  Too